# 03 — Treino real + exportação

Pipeline completo no Mac (MPS) / CUDA:

1. CodeXGLUE → normalizado (pula se já existir)
2. SFT real em `dataset/processed/sft_real/` (pula se já existir)
3. Treino LoRA → `models/qwen3-legacy-doc-lora-real/`
4. Export → `models/export/qwen2.5-1.5b-celx/`
5. Teste de inferência

**Kernel obrigatório:** `Select Kernel` → **Python (Celx .venv)**  
caminho: `/Users/wolfx/Documents/Dev/Celx/.venv/bin/python`

**Ordem:** rode as células de cima para baixo. As seções 5–7 redefinem os paths sozinhas (ok após restart do kernel).

**Acompanhar (outro Terminal):**
```bash
cd /Users/wolfx/Documents/Dev/Celx
source .venv/bin/activate
python scripts/watch_training.py
```

**Alternativa só no terminal:**
```bash
bash scripts/retrain_real.sh
```

Config: `configs/train_real.yaml`


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

EXPECTED = Path("/Users/wolfx/Documents/Dev/Celx/.venv/bin/python").resolve()
current = Path(sys.executable).resolve()
print("Python:", current)
if current != EXPECTED and ".venv" not in str(current):
    raise RuntimeError(
        "Kernel ERRADO.\n"
        "Select Kernel → Python (Celx .venv)\n"
        f"Esperado: {EXPECTED}\nAtual: {current}"
    )

ROOT = Path("/Users/wolfx/Documents/Dev/Celx").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = str(ROOT)
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.0")

import torch

if torch.cuda.is_available():
    BACKEND = "cuda"
elif torch.backends.mps.is_available():
    BACKEND = "mps"
else:
    BACKEND = "cpu"

CONFIG = ROOT / "configs" / "train_real.yaml"
OUTPUT_DIR = ROOT / "models" / "qwen3-legacy-doc-lora-real"
EXPORT_DIR = ROOT / "models" / "export" / "qwen2.5-1.5b-celx"
SFT_DIR = ROOT / "dataset" / "processed" / "sft_real"
CODEX_DIR = ROOT / "dataset" / "processed" / "codexglue"
MONITOR = ROOT / "outputs" / "training_real"
MONITOR.mkdir(parents=True, exist_ok=True)

print("Backend:", BACKEND, "| torch", torch.__version__)
print("Config:", CONFIG)
print("Kernel OK")
if BACKEND == "mps":
    print("MPS: feche apps pesados durante o treino (8GB).")
elif BACKEND == "cpu":
    print("Aviso: CPU será muito lento para treino real.")


In [ ]:
need = []
for mod, spec in [
    ("transformers", "transformers>=4.51"),
    ("accelerate", "accelerate>=1.0"),
    ("datasets", "datasets>=3.0"),
    ("peft", "peft>=0.14"),
    ("trl", "trl>=0.15"),
    ("yaml", "PyYAML>=6.0"),
    ("pandas", "pandas>=2.0"),
    ("tensorboard", "tensorboard>=2.14"),
]:
    try:
        __import__(mod)
    except ModuleNotFoundError:
        need.append(spec)

if need:
    print("Instalando:", ", ".join(need))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
else:
    print("Deps OK (já instaladas).")

import legacy_doc
print("legacy_doc", legacy_doc.__version__)


## 1) CodeXGLUE (download + normalização)

Gera `dataset/processed/codexglue/`. **Pula** se o manifest já existir.


In [ ]:
import json

codex_manifest = CODEX_DIR / "manifest.json"
if codex_manifest.exists():
    print("CodeXGLUE já preparado — pulando download.")
    print(json.dumps(json.loads(codex_manifest.read_text()), indent=2)[:1200])
else:
    print("Baixando/normalizando CodeXGLUE (pode demorar)...")
    subprocess.run(
        [sys.executable, "scripts/prepare_dataset.py", "--config", str(CONFIG)],
        check=True,
        cwd=ROOT,
    )
    print(json.dumps(json.loads(codex_manifest.read_text()), indent=2)[:1200])


## 2) SFT real (não smoke)

Escreve em `dataset/processed/sft_real/`. Exige ≥ 100 exemplos.
**Pula** se train já tiver samples suficientes.


In [ ]:
from datasets import load_from_disk

def sft_counts():
    if not (SFT_DIR / "train").exists():
        return 0, 0
    return (
        len(load_from_disk(str(SFT_DIR / "train"))),
        len(load_from_disk(str(SFT_DIR / "validation"))),
    )

train_n, val_n = sft_counts()
if train_n >= 100:
    print(f"SFT real já existe — Train: {train_n} | Validation: {val_n}")
else:
    print("Construindo SFT real a partir do CodeXGLUE...")
    subprocess.run(
        [sys.executable, "scripts/build_sft_from_codexglue.py", "--config", str(CONFIG)],
        check=True,
        cwd=ROOT,
    )
    train_n, val_n = sft_counts()
    print(f"Train: {train_n} | Validation: {val_n}")

assert train_n >= 100, f"Train pequeno demais ({train_n}); abortando."
manifest = SFT_DIR / "manifest.json"
if manifest.exists():
    print(manifest.read_text(encoding="utf-8"))
print("SFT real OK — não é smoke.")


## 3) Monitoramento

Em **outro** Terminal:

```bash
cd /Users/wolfx/Documents/Dev/Celx
source .venv/bin/activate
python scripts/watch_training.py
```

Arquivos: `outputs/training_real/STATUS.txt`, `live.json`, `metrics.csv`, `train.log`


In [ ]:
logdir = MONITOR / "runs"
logdir.mkdir(parents=True, exist_ok=True)
print("STATUS:", MONITOR / "STATUS.txt")
print("live:  ", MONITOR / "live.json")
print("log:   ", MONITOR / "train.log")
print("Watch:  python scripts/watch_training.py")


## 4) Treinar LoRA (real)

`--resume` continua de checkpoint se existir. Adapter final: `models/qwen3-legacy-doc-lora-real/`


In [ ]:
live_json = MONITOR / "live.json"
metrics_csv = MONITOR / "metrics.csv"
train_log = MONITOR / "train.log"
error_txt = MONITOR / "error.txt"

cmd = [
    sys.executable, "scripts/train_lora.py",
    "--config", str(CONFIG),
    "--backend", BACKEND,
    "--output-dir", str(OUTPUT_DIR),
    "--resume",
]
print("Comando:", " ".join(cmd))
print("Log:", train_log)

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT)
env["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

with train_log.open("a", encoding="utf-8") as logf:
    logf.write("\n===== notebook 03 start =====\n")
    logf.flush()
    proc = subprocess.Popen(
        cmd, cwd=ROOT, env=env, stdout=logf, stderr=subprocess.STDOUT,
    )

try:
    last_status = ""
    while proc.poll() is None:
        status_path = MONITOR / "STATUS.txt"
        if status_path.exists():
            text = status_path.read_text(encoding="utf-8").strip()
            if text and text != last_status:
                print(text)
                print("-" * 40)
                last_status = text
        time.sleep(20)
except KeyboardInterrupt:
    print("Interrompendo treino...")
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()
    raise

rc = proc.wait()
if rc != 0:
    if error_txt.exists():
        print(error_txt.read_text(encoding="utf-8"))
    print("\n".join(train_log.read_text(encoding="utf-8").splitlines()[-40:]))
    raise RuntimeError(f"Treino falhou: {rc}. Veja {train_log}")

assert (OUTPUT_DIR / "adapter_config.json").exists(), OUTPUT_DIR
print("Treino real concluído →", OUTPUT_DIR)


## 5) Métricas e arquivos do adapter

Pode rodar mesmo após restart do kernel (paths definidos abaixo).


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/wolfx/Documents/Dev/Celx").resolve()
OUTPUT_DIR = ROOT / "models" / "qwen3-legacy-doc-lora-real"
metrics_csv = ROOT / "outputs" / "training_real" / "metrics.csv"

assert metrics_csv.exists(), (
    f"Sem metrics.csv em {metrics_csv}.\n"
    "Rode a seção 4 (treino) antes — ou o treino ainda não gerou logs."
)
df = pd.read_csv(metrics_csv)
print(df.tail(30).to_string(index=False))

try:
    import matplotlib.pyplot as plt
    plot = df.dropna(subset=["loss"])
    if not plot.empty:
        plt.figure(figsize=(8, 4))
        plt.plot(plot["step"], plot["loss"], label="train loss")
        ev = df.dropna(subset=["eval_loss"])
        if not ev.empty:
            plt.plot(ev["step"], ev["eval_loss"], label="eval loss")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.title("Treino real")
        plt.show()
except Exception as exc:
    print("Plot opcional:", exc)

assert (OUTPUT_DIR / "adapter_config.json").exists(), (
    f"Adapter não encontrado em {OUTPUT_DIR}. Treino ainda não terminou."
)
print("Arquivos do adapter:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f"- {p.name} ({p.stat().st_size // 1024} KB)")
metrics_path = OUTPUT_DIR / "metrics.json"
if metrics_path.exists():
    print(metrics_path.read_text(encoding="utf-8")[:1500])


## 6) Exportar para uso

Copia o adapter para `models/export/qwen2.5-1.5b-celx/`. Sem `--merge` no M1 8GB.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path("/Users/wolfx/Documents/Dev/Celx").resolve()
OUTPUT_DIR = ROOT / "models" / "qwen3-legacy-doc-lora-real"
EXPORT_DIR = ROOT / "models" / "export" / "qwen2.5-1.5b-celx"

assert (OUTPUT_DIR / "adapter_config.json").exists(), (
    f"Adapter ainda não existe em {OUTPUT_DIR}. Rode a seção 4 (treino)."
)

subprocess.run(
    [
        sys.executable, "scripts/export_model.py",
        "--adapter", str(OUTPUT_DIR),
        "--output", str(EXPORT_DIR),
    ],
    check=True,
    cwd=ROOT,
)
print("Export:", EXPORT_DIR)
readme = EXPORT_DIR / "README.md"
if readme.exists():
    print(readme.read_text(encoding="utf-8"))


## 7) Testar o modelo exportado


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path("/Users/wolfx/Documents/Dev/Celx").resolve()
EXPORT_DIR = ROOT / "models" / "export" / "qwen2.5-1.5b-celx"
CONFIG = ROOT / "configs" / "train_real.yaml"
adapter_path = EXPORT_DIR / "adapter"

assert (adapter_path / "adapter_config.json").exists(), (
    f"Export ainda não existe em {adapter_path}. Rode a seção 6."
)

subprocess.run(
    [
        sys.executable, "scripts/document_code.py",
        "--file", str(ROOT / "dataset/examples/calcula_total.py"),
        "--language", "python",
        "--adapter", str(adapter_path),
        "--config", str(CONFIG),
    ],
    check=True,
    cwd=ROOT,
)
print("OK — modelo exportado em", EXPORT_DIR)
print(
    "Uso: python scripts/document_code.py "
    "--file dataset/examples/calcula_total.py "
    "--language python "
    f"--adapter {adapter_path}"
)
